# Track 2 — Training from Scratch Orchestrator

Thin orchestrator only. All real logic lives in `slm_from_scratch/scripts/*.py`
(agent-editable, ordinary `.py` modules). Cells below just **sync code**, **install
deps**, and **call into those scripts**.

**Before running the sync cell:** after any local agent edit you MUST commit and push
(`git add -A && git commit -m ... && git push`) so the remote kernel pulls your
latest code. Re-running the sync cell picks up new edits.

> **GPU tip:** If you already have a Colab session running for Track 1, reuse it
> via *Auto Connect* — saves GPU queue wait time. If starting fresh, connect a
> T4/A100 runtime before running any cells.

## Cell 1 — Sync: clone or pull latest code

In [ ]:
!git clone https://github.com/DevaNandanJS/Benchmarking-LLM-fine-tuning-vs-training-from-scratch-using-the-same-dataset.git llm_task 2>/dev/null || (cd llm_task && git pull)
%cd llm_task

## Cell 2 — Install dependencies

Key Track 2 dep: `tokenizers` (Hugging Face Rust-backed BPE trainer).  
`torch` is NOT in requirements.txt — Colab ships a CUDA-enabled version already.

In [ ]:
!pip install -q -r requirements.txt
# Freeze the exact versions to slm_from_scratch/logs/environment.txt
# (reproducibility lock — commit this file back to the repo)
import os
os.makedirs('slm_from_scratch/logs', exist_ok=True)
!pip freeze > slm_from_scratch/logs/environment.txt
print('Dependency snapshot written to slm_from_scratch/logs/environment.txt')

## Cell 3 — Hardware check

Confirms a GPU is attached and logs key info. Script exits non-zero if no GPU found —
do NOT proceed with training cells if this cell fails.

In [ ]:
import torch
assert torch.cuda.is_available(), 'No GPU detected — connect a Colab GPU runtime first'
props = torch.cuda.get_device_properties(0)
print('GPU:         ', torch.cuda.get_device_name(0))
print('VRAM (GB):   ', round(props.total_memory / 1e9, 2))
print('torch:       ', torch.__version__)
import tokenizers
print('tokenizers:  ', tokenizers.__version__)

---

## Phase 1 — Custom Tokenizer Training

**Goal:** sweep vocab sizes 256 / 1024 / 4096, compute fertility for each,
select the best size using a diminishing-returns criterion, save the final
tokenizer files, and write `configs/tokenizer_choice.md` with actual numbers.

**Outputs produced:**
- `slm_from_scratch/eval/vocab_sweep.csv`
- `slm_from_scratch/tokenizer/vocab.json` + `merges.txt`
- `slm_from_scratch/configs/tokenizer_choice.md`
- `slm_from_scratch/configs/run_phase1_tokenizer.json`
- `slm_from_scratch/tokenizer_candidates/vocab{256,1024,4096}/` (all candidates saved)

**After this cell:** commit + push all of the above so they are version-controlled.

In [ ]:
!python slm_from_scratch/scripts/train_tokenizer.py

### Inspect sweep results

In [ ]:
import pandas as pd
df = pd.read_csv('slm_from_scratch/eval/vocab_sweep.csv')
print(df.to_string(index=False))
print()
print(open('slm_from_scratch/configs/tokenizer_choice.md').read())

---

## Phase 2 -- Dataset Construction

**Prerequisite:** Phase 1 must have run on Colab and been committed + pushed.
Specifically, `slm_from_scratch/tokenizer/vocab.json` and `merges.txt` must exist
in the repo after your `git pull`. If they are missing, run Phase 1 first.

**Inputs:**
- `slm_from_scratch/tokenizer/vocab.json` + `merges.txt` (from Phase 1)
- `data/extracted/document_clean.txt` (from Track 1 Phase 1)

**Outputs produced:**
- `data/processed/slm_train.pt`
- `data/processed/slm_val.pt`
- `data/processed/track2_dataset_stats.json`
- `slm_from_scratch/configs/run_phase2_dataset.json`
- `slm_from_scratch/configs/split_strategy.md`

**After this cell:** commit + push all outputs so they are version-controlled.

In [ ]:
!python slm_from_scratch/scripts/build_dataset.py

### Inspect Phase 2 results

In [ ]:
import json as _json
import torch

stats = _json.load(open("data/processed/track2_dataset_stats.json"))
print("=== Track 2 Dataset Stats ===")
for k, v in stats.items():
    sv = str(v)
    print(f"  {k}: {sv[:80]}..." if len(sv) > 80 else f"  {k}: {sv}")

train = torch.load("data/processed/slm_train.pt", weights_only=True)
val   = torch.load("data/processed/slm_val.pt",   weights_only=True)
print("\ntrain input_ids shape:", tuple(train["input_ids"].shape))
print("val   input_ids shape:", tuple(val["input_ids"].shape))
print("chars/window:", stats["chars_per_window"], "raw chars per training example")

print("\n=== Split Strategy (excerpt) ===")
print(open("slm_from_scratch/configs/split_strategy.md").read()[:800])


---

## Phase 3 — Model Architecture Implementation

**Goal:** Build and validate a from-scratch decoder-only Transformer. Reads
`vocab_size` from the Phase 1 tokenizer and `block_size` from Phase 2 stats
automatically; falls back to `vocab_size=1024, block_size=256` if those files
are not yet present (safe for local smoke-testing).

**Prerequisite:** Phase 1 and Phase 2 must have run on Colab and been committed
+ pushed so `git pull` in Cell 1 picks up `slm_from_scratch/tokenizer/vocab.json`
and `data/processed/track2_dataset_stats.json`. If those are missing the script
falls back to defaults and prints a warning — the unit tests still run.

**Outputs (written on test success):**
- `slm_from_scratch/configs/run_phase3_model.json` — architecture config dump
- `slm_from_scratch/configs/trainable_params.json` — per-component param count

**After this cell:** Verify all 8 tests pass, commit + push the two JSON
artifacts, then proceed to Phase 4.

In [ ]:
# Cell 3b — Run unit tests + dump architecture artifacts
# Runs all 8 unit tests unconditionally; writes JSON artifacts only on success.
# If any test raises AssertionError, execution stops here and Cell 3c will
# FileNotFoundError — fix the failing test before proceeding.
!python slm_from_scratch/scripts/model.py

In [ ]:
# Cell 3c — Inspect architecture config and parameter breakdown
# These files are written by Cell 3b AFTER all unit tests pass.
# FileNotFoundError here means Cell 3b failed — check its output above.
import json as _json

cfg = _json.load(open('slm_from_scratch/configs/run_phase3_model.json'))
print('=== Phase 3 Model Config ===')
for k, v in cfg.items():
    print(f'  {k}: {v}')

params = _json.load(open('slm_from_scratch/configs/trainable_params.json'))
print('\n=== Parameter Count ===')
for k, v in params.items():
    if isinstance(v, int):
        print(f'  {k}: {v:,}')
    else:
        print(f'  {k}: {v}')

---

## Phase 4 — Training Loop

**Prerequisites:** Phases 1–3 must have run on Colab and all outputs committed + pushed.
Specifically, these files must exist after `git pull` in Cell 1:
- `slm_from_scratch/tokenizer/vocab.json` (Phase 1)
- `data/processed/slm_train.pt` and `slm_val.pt` (Phase 2)
- `slm_from_scratch/configs/trainable_params.json` (Phase 3)

**Sweep runs (one cell each):**
| Run | n_layer | n_embd | lr | Sweep axis |
|---|---|---|---|---|
| `small` | 4 | 128 | 3e-4 | Architecture |
| `base` | 6 | 192 | 3e-4 | Architecture |
| `base_highlr` | 6 | 192 | 6e-4 | Learning rate |

**Outputs produced (commit after all 3 runs):**
- `slm_from_scratch/configs/run_phase4_<run>.json` — config dump (pre-training)
- `slm_from_scratch/logs/<run>/metrics.jsonl` — step-level loss/LR log
- `slm_from_scratch/checkpoints/best_val/<run>/best_val.pt` — best checkpoint
- `slm_from_scratch/checkpoints/best_val/<run>/best_val_config.json`
- `slm_from_scratch/checkpoints/last/<run>/last_ckpt.pt` — final-step audit artifact
- `slm_from_scratch/eval/sweep_results.csv` — one row per run

> **After all 3 runs:** `git add -A && git commit -m 'Phase 4: training complete' && git push`

### Cell 4b — Smoke-test (local pre-flight, CPU, ~5 seconds)

Validates all code paths (shapes, loss, gradients, LR schedule, `evaluate()`).
Does **not** write checkpoints or update `sweep_results.csv`.

In [ ]:
# Smoke-test: 4 chunks, 5 steps, CPU fp32.
# Run this locally before pushing to Colab to catch shape/dtype bugs early.
!python slm_from_scratch/scripts/train.py --run base --smoke-test

### Cell 4c — Run `small` (architecture sweep: 4 layers, 128-embd, lr=3e-4)

**Expected:** initial loss ≈ ln(1024) ≈ 6.93 (random-model baseline), then descent.
Smaller architecture — expect faster overfitting than `base`.

In [ ]:
!python slm_from_scratch/scripts/train.py --run small

### Cell 4d — Run `base` (architecture sweep: 6 layers, 192-embd, lr=3e-4)

In [ ]:
!python slm_from_scratch/scripts/train.py --run base

### Cell 4e — Run `base_highlr` (LR sweep: same arch as `base`, lr=6e-4)

Same architecture as `base` — isolates the effect of a 2x higher learning rate.
Expect faster initial descent but potentially earlier/worse overfitting.

In [ ]:
!python slm_from_scratch/scripts/train.py --run base_highlr

### Cell 4f — Inspect sweep results

In [ ]:
import pandas as pd
df = pd.read_csv('slm_from_scratch/eval/sweep_results.csv')
print(df.to_string(index=False))
print(f"\nBest run: {df.loc[df['best_val_loss'].idxmin(), 'run_name']}  "
      f"(best_val_loss={df['best_val_loss'].min():.4f})")

### Cell 4g — Inspect best checkpoint configs

In [ ]:
import json as _json, os
for run in ['small', 'base', 'base_highlr']:
    cfg_path = f'slm_from_scratch/checkpoints/best_val/{run}/best_val_config.json'
    if os.path.exists(cfg_path):
        cfg = _json.load(open(cfg_path))
        print(f"\n=== {run} best checkpoint ===")
        for k in ['run_name', 'n_layer', 'n_embd', 'learning_rate',
                  'best_val_loss', 'best_val_step', 'total_steps', 'timestamp']:
            print(f"  {k}: {cfg.get(k, 'N/A')}")
    else:
        print(f"\n[{run}] best_val_config.json not found -- run Cell 4c/4d/4e first")

---

## Phase 5 — Quantitative Evaluation

**Prerequisites:** Phase 4 (all 3 sweep runs) must be complete and committed.
Specifically, these files must exist after `git pull`:
- `slm_from_scratch/eval/sweep_results.csv` (written by train.py)
- `slm_from_scratch/checkpoints/best_val/<run>/best_val.pt` (best checkpoint)
- `data/processed/slm_val.pt` (val tensors from Phase 2)
- `data/processed/track2_dataset_stats.json` (must contain `split_boundary_token_idx`)

**Outputs produced:**
- `slm_from_scratch/eval/loss_curve.png` -- train/val curves for all 3 sweep runs
- `slm_from_scratch/eval/final_metrics.json` -- CE loss, perplexity, BPB
- `slm_from_scratch/eval/loss_curve_interpretation.md` -- templated interpretation

**After this cell:** commit all three outputs, then run Phase 6.

In [ ]:
# Cell 5a -- Smoke-test (CPU, no checkpoint or real data needed)
# Verifies the BPB accumulation logic, plotting, and JSON writing.
!python slm_from_scratch/scripts/eval.py --smoke-test

In [ ]:
# Cell 5b -- Run full quantitative evaluation (GPU required)
# Reads sweep_results.csv to auto-detect the best run.
# Pass --run <name> to override: !python ... --run base
!python slm_from_scratch/scripts/eval.py

In [ ]:
# Cell 5c -- Inspect evaluation outputs
import json as _json

metrics = _json.load(open('slm_from_scratch/eval/final_metrics.json'))
print('=== Track 2 Final Metrics ===')
for k, v in metrics.items():
    sv = str(v)
    print(f'  {k}: ' + (sv[:100] + '...' if len(sv) > 100 else sv) + '')

bpb_gap = metrics['bpb'] - 1.309722
print(f"\nBPB = {metrics['bpb']}  (Track 1 = 1.309722  gap = {bpb_gap:+.6f})")
print('\n=== Loss Curve Interpretation (excerpt) ===')
print(open('slm_from_scratch/eval/loss_curve_interpretation.md').read())


---

## Phase 6 -- Qualitative Evaluation (Generation)

**Prerequisites:** Phase 5 must have completed and its outputs committed.
The best checkpoint must exist at:
`slm_from_scratch/checkpoints/best_val/<best_run>/best_val.pt`

**Prompts:** same 8 prompts as Track 1 (controlled comparison).

**Outputs produced:**
- `slm_from_scratch/generations/slm_samples.md` -- 8 prompts x 2 decoding modes

**After this cell:** commit slm_samples.md, review annotations manually,
revise labels where the heuristic is wrong, then run Phase 7.

In [ ]:
# Cell 6a -- Smoke-test (CPU, no checkpoint needed)
# Verifies generate() determinism, annotation logic, markdown writing.
!python slm_from_scratch/scripts/generate.py --smoke-test

In [ ]:
# Cell 6b -- Run qualitative generation (GPU recommended)
# Uses the same 8 prompts as Track 1 -- controlled comparison.
# Sampling: T=0.8, top_p=0.9  |  Greedy: argmax (T=0)
!python slm_from_scratch/scripts/generate.py

In [ ]:
# Cell 6c -- Preview generated samples
print(open('slm_from_scratch/generations/slm_samples.md').read())


---

## Phase 7 -- Cross-Track Comparison

**Prerequisites:** Phase 5 AND Phase 6 must have completed and been committed.
Also requires `shared_eval/finetuning_final_metrics.json` (already committed).

**Outputs produced:**
- `shared_eval/slm_final_metrics.json` -- copy of Phase 5 metrics
- `shared_eval/slm_loss_curve.png` -- copy of Phase 5 curve
- `shared_eval/comparison_notes.md` -- fully populated (no TBD)

**DoD assertion:** the script hard-fails if any "TBD" remains in comparison_notes.md.

**After this cell:** commit shared_eval/ outputs.
Phase 7 is the final phase -- the project is complete.

In [ ]:
# Cell 7a -- Smoke-test
!python slm_from_scratch/scripts/compare.py --smoke-test

In [ ]:
# Cell 7b -- Run cross-track comparison (CPU, fast)
# Copies artefacts to shared_eval/ and writes comparison_notes.md.
!python slm_from_scratch/scripts/compare.py

In [ ]:
# Cell 7c -- Preview comparison notes
print(open('shared_eval/comparison_notes.md').read())
